---

### [UCI](https://archive.ics.uci.edu/)
* [bank marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)
* [Iris](https://archive.ics.uci.edu/dataset/53/iris)


_Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306. \| CC-BY 4.0_

_Fisher, R. (1936). Iris [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56C76. \| CC-BY 4.0_

---

In [ ]:
!pip install ucimlrepo

Using Python 3.12.12 environment at: /usr
Resolved 8 packages in 180ms
Prepared 1 package in 11ms
Installed 1 package in 1ms
 + ucimlrepo==0.0.7


In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id = 222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)


取得範例資料

In [ ]:
# 直接用 magic command，或貼到 terminal 執行
!curl -o bank_marketing https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
# !curl -o iris https://archive.ics.uci.edu/static/public/53/iris.zip
!unzip bank_marketing
!unzip bank.zip

!rm -rf sample_data bank_marketing.zip bank.zip bank_marketing


裝套件

In [ ]:
# !pip install numpy pandas scikit-learn xgboost matplotlib seaborn

In [ ]:
import pandas as pd
df_full = pd.read_csv('bank-full.csv', sep = ';')
df_full

In [ ]:
df_full['job'].value_counts()
df_full['housing']

沒想法？  
視覺化看看

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for plots 選圖風
sns.set_style('whitegrid')

# Get numerical and categorical columns 取得數值、分類型的欄位
numerical_cols = df_full.select_dtypes(include = ['int64', 'float64']).columns
categorical_cols = df_full.select_dtypes(include = ['object']).columns

# Plot histograms for numerical features 把數值型的畫成直方圖
print('Plotting Histograms for Numerical Features...')
plt.figure( figsize = (15, 10) )
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 6, i + 1) # Adjust subplot grid as needed
    sns.histplot(df_full[col], kde = True) # KDE: Kernel Density Estimate
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
plt.show()

# Plot countplots for categorical features 把類別型點出來
print('\nPlotting Countplots for Categorical Features...')
plt.figure( figsize = (15, 10) )
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 6, i + 1) # Adjust subplot grid as needed
    sns.countplot(y = df_full[col], order = df_full[col].value_counts().index)
    plt.title(f'Count of {col}')
    plt.tight_layout()
plt.show()

In [ ]:
# df_bank = pd.read_csv('bank.csv', sep = ';') # 10%
# df_bank

把 label 轉換成數值

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df_full['housing'] = le.fit_transform(df_full['housing'])
df_full['y'] = le.fit_transform(df_full['y'])
df_full


拆 data、label

In [ ]:
import numpy as np
label = df_full['y'].to_numpy()
label
data = df_full['housing'].to_numpy()
data


In [ ]:
# regress housing over y 用「有無房貸」對 y 做回歸
from sklearn.linear_model import LinearRegression
# from sklearn.linear_model import LogisticRegression

reg = LinearRegression()
reg.fit(data.reshape(-1, 1), label) # sklearn regression requires 2D regressor input sklearn 的回歸要求 2D 輸入
reg.score(data.reshape(-1, 1), label)
reg.coef_
reg.intercept_
reg.predict( np.array([1]).reshape(-1, 1) )

In [ ]:
# RMSE
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_pred = reg.predict( data.reshape(-1, 1) )
mse = mean_squared_error(label, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(label, y_pred)

print(f'Mean Squared Error: {mse:.4f}') # MSE
print(f'Root Mean Squared Error: {rmse:.4f}') # RMSE
print(f'Mean Absolute Error: {mae:.4f}') # MAE

鬧完了，來做正經的 (？)

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# load data 載入資料
df_full = pd.read_csv('bank-full.csv', sep = ';')

# transform categorical data into numerical ones 將類別型變數轉換為數值型
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
for col in categorical_cols:
    df_full[col] = LabelEncoder().fit_transform(df_full[col])

# transform label too 將標籤 y 轉換為數值型
df_full['y'] = LabelEncoder().fit_transform(df_full['y'])

# prepare features and label 分割特徵和標籤
X = df_full.drop('y', axis = 1)
y = df_full['y']

# split training and validation dataset 分割訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# standardisation 標準化特徵
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


資料探勘，降維

In [ ]:
# PCA
from sklearn.decomposition import PCA

# to 2D
pca = PCA(n_components = 2)
X_train_pca = pca.fit_transform(X_train_scaled)

print(f'PC1 and PC2: {pca.explained_variance_ratio_}')




In [ ]:
import matplotlib.pyplot as plt

# get PC1 and PC2 explained variance ratio 取得 PC1 和 PC2 的可解釋方差比例
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

# plot it out 畫出來看看
plt.figure( figsize = (8, 6) )
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c = y_train, cmap = 'viridis', alpha = 0.8)
plt.xlabel(f'Principal Component 1 ({pc1_var:.1f}%)')
plt.ylabel(f'Principal Component 2 ({pc2_var:.1f}%)')
plt.title('2D PCA of Bank Marketing Data')
plt.colorbar(label = 'Target Variable (y)') # 0 or 1, housing or not
plt.grid(True)
plt.show()

In [ ]:
# Elastic Net
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LogisticRegression

# elastic_net = LogisticRegression(penalty = 'elasticnet', solver = 'saga', l1_ratio = 0.5, random_state = 42)
elastic_net = ElasticNet(alpha = 0.1, l1_ratio = 0.5, random_state = 42)
elastic_net.fit(X_train_scaled, y_train)

train_score = elastic_net.score(X_train_scaled, y_train)
test_score = elastic_net.score(X_test_scaled, y_test)

print(f'Training score: {train_score:.4f}')
print(f'validation score: {test_score:.4f}')




In [ ]:
# get feature names 取得特徵名稱
feature_names = X.columns

# get the coefficients 取得 Elastic Net 的係數
coefficients = elastic_net.coef_

# create a dataframe using the weight of each feature 建立 DataFrame 顯示每個特徵的權重
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# for logistic regression
# coef_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Coefficient': coefficients[0] # Access the first row to make it 1-dimensional
# })


# rerank according to absolute coefficients 依權重絕對值排序
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df_sorted = coef_df.sort_values(by = 'Abs_Coefficient', ascending = False)

print(coef_df_sorted)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure( figsize = (10, 6) )
sns.barplot(data = coef_df_sorted, x = 'Coefficient', y = 'Feature', palette = 'viridis', hue = 'Feature')
plt.title('Elastic Net Coefficients')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.show()

In [ ]:
from sklearn.svm import SVC

# Support Vector Classifier 支援向量分類器
svc_model = SVC(random_state = 42)
svc_model.fit(X_train_scaled, y_train)

train_score_svc = svc_model.score(X_train_scaled, y_train)
test_score_svc = svc_model.score(X_test_scaled, y_test)

print(f'SVC Training score: {train_score_svc:.4f}')
print(f'SVC Validation score: {test_score_svc:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix
# import matplotlib.pyplot as plt
# import seaborn as sns

# Make predictions on the test (validation)set
y_pred = svc_model.predict(X_test_scaled)

# Compute the confusion matrix 算混淆矩陣
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix 畫出來
plt.figure( figsize = (8, 6) )
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues',
            xticklabels = ['Predicted No', 'Predicted Yes'],
            yticklabels = ['Actual No', 'Actual Yes'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for SVC Model')
plt.show()

In [ ]:
# 算其他指標
from sklearn.metrics import f1_score, recall_score, precision_score

y_pred = svc_model.predict(X_test_scaled)

f1 = f1_score(y_test, y_pred)

# Recall (Sensitivity)
recall = recall_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

print(f'F1-score: {f1:.4f}')
print(f'Recall (Sensitivity): {recall:.4f}')
print(f'Precision: {precision:.4f}')

In [ ]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

# Initialise and train the Gaussian Naive Bayes model 建立、訓練分類器
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Evaluate the model 評估模型
train_score_nb = nb_model.score(X_train_scaled, y_train)
test_score_nb = nb_model.score(X_test_scaled, y_test)

print(f'Naive Bayes Training score: {train_score_nb:.4f}')
print(f'Naive Bayes Validation score: {test_score_nb:.4f}')

In [ ]:
# 交叉驗證
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation for Naive Bayes 交叉驗證
cv_scores = cross_val_score(nb_model, X_train_scaled, y_train, cv = 5)

print(f'Naive Bayes Cross-validation scores: {cv_scores}')
print(f'Naive Bayes Mean cross-validation score: {cv_scores.mean():.4f}')

Decision tree and ensemble learning  
決策樹與集成學習

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators = 100, random_state = 42) # 100 -> prone to overfitting
random_forest.fit(X_train_scaled, y_train)

train_score = random_forest.score(X_train_scaled, y_train)
test_score = random_forest.score(X_test_scaled, y_test)

print(f'Random Forest training score: {train_score:.4f}')
print(f'Random Forest validation score: {test_score:.4f}')

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators = 10, random_state = 42)
xgb.fit(X_train_scaled, y_train)

train_score = xgb.score(X_train_scaled, y_train)
test_score = xgb.score(X_test_scaled, y_test)

print(f'XGBoost training score: {train_score:.4f}')
print(f'XGBoost validation score: {test_score:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# XGBoost
y_pred = xgb.predict(X_test_scaled)

print('confusion matrix')
print( confusion_matrix(y_test, y_pred) )

print('report')
# support: number of samples supporting the results
print( classification_report(y_test, y_pred) )